In [1]:
import ctypes
try:
    ctypes.CDLL('/usr/lib/aarch64-linux-gnu/libgomp.so.1', mode=ctypes.RTLD_GLOBAL)
except OSError:
    pass

import os
import sys
sys.path.append('/usr/local/lib')
sys.path.append('/usr/local/lib/python3.6/pyrealsense2')

import cv2
import torch
import numpy as np
import math
import time
import csv
import pyrealsense2 as rs
import ipywidgets as widgets
from IPython.display import display
from jetcam.utils import bgr8_to_jpeg
from jetcam.csi_camera import CSICamera
from jetracer.nvidia_racecar import NvidiaRacecar
from torch2trt import TRTModule
from utils import preprocess

def quaternion_yaw(qx, qy, qz, qw):
    siny_cosp = 2 * (qw * qy + qz * qx)
    cosy_cosp = 1 - 2 * (qx * qx + qy * qy)
    return math.atan2(siny_cosp, cosy_cosp)

model_trt = TRTModule()
model_trt.load_state_dict(torch.load('road_following_model_merged_0337t_tanh_norbert_laptop_trt.pth'))

car = NvidiaRacecar()
camera = CSICamera(width=224, height=224, capture_fps=30)
camera.running = True

pipe = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.pose)
pipe.start(config)

print("Sprzęt, model TRT oraz T265 zainicjalizowane pomyślnie!")

Sprzęt, model TRT oraz T265 zainicjalizowane pomyślnie!


In [2]:
camera_widget = widgets.Image(format='jpeg', width=224, height=224)
display(camera_widget)

Image(value=b'', format='jpeg', height='224', width='224')

In [ ]:
STEERING_GAIN = -1.00
STEERING_BIAS = 0.03
THROTTLE = -0.45  

car.throttle = THROTTLE

SAVE_INTERVAL = 0.1
last_save_time = 0
vel = 0.0  # Zmienna zainicjalizowana przed pętlą

# Otwarcie pliku raz na całą sesję
csv_file_path = 'T265_tracking_data.csv'
file_exists = os.path.exists(csv_file_path)
csv_file = open(csv_file_path, 'a', newline='', encoding='utf-8')
writer = csv.writer(csv_file)

if not file_exists:
    writer.writerow(['X (m)', 'Z (m)', 'YAW (deg)', 'VEL (m/s)'])
    csv_file.flush()

try:
    while True:
        current_time = time.time()
        
        # Odbiór klatek T265
        frames = pipe.poll_for_frames()
        if frames:
            pose_frame = frames.get_pose_frame()
            if pose_frame:
                data = pose_frame.get_pose_data()
                
                # Sprzętowa prędkość liniowa z T265 (bez liczenia sqrt w Pythonie)
                v = data.velocity
                vel = round(math.sqrt(v.x**2 + v.y**2 + v.z**2), 2)
                
                if current_time - last_save_time >= SAVE_INTERVAL:
                    x_m = round(data.translation.x, 2)
                    z_m = round(-data.translation.z, 2)
                    
                    qx, qy, qz, qw = data.rotation.x, data.rotation.y, data.rotation.z, data.rotation.w
                    yaw_deg = round(math.degrees(quaternion_yaw(qx, qy, qz, qw)), 2)
                    
                    writer.writerow([x_m, z_m, yaw_deg, vel])
                    last_save_time = current_time

        # Inferencja i sterowanie
        image = camera.value
        if image is None:
            continue
            
        cv2.putText(image, f"{vel} m/s", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        camera_widget.value = bgr8_to_jpeg(image)

        image_preprocessed = preprocess(image).half()
        output = model_trt(image_preprocessed).detach().cpu().numpy().flatten()

        x = float(output[0])
        car.steering = x * STEERING_GAIN + STEERING_BIAS

except KeyboardInterrupt:
    print("\nZatrzymano przez użytkownika.")
finally:
    csv_file.close()
    pipe.stop()
    car.throttle = 0.0
    car.steering = 0.0
    print("Pojazd zatrzymany, potok T265 zamknięty.")